In [204]:
import os
import sys
sys.path.append('..')
from scripts.utils import mean_absolute_error, load_dataset
import pandas as pd

In [205]:
dam_test = load_dataset("../dataset/dam/data_test.csv",'Date')
dam_train = load_dataset("../dataset/dam/data_train.csv",'Date')
imb_test = load_dataset("../dataset/imb/data_test.csv",'timestamp')
imb_train = load_dataset("../dataset/imb/data_train.csv",'timestamp')

In [206]:
forecast_folder = '../scripts/raw_forecast'

## DAM raw

In [207]:
dataset = 'dam'
timestamp_col_parse = 'Date'
_forecast_folder = os.path.join(forecast_folder,dataset)
point_forecast_df = {}
upper_df = {}
lower_df = {}
for fname in os.listdir(_forecast_folder):
    if 'csv' in fname and 'lora' not in fname:
        model_name = fname.replace('.csv','')
        raw_forecast = pd.read_csv(os.path.join(_forecast_folder, fname), parse_dates=[timestamp_col_parse])
        point_forecast = raw_forecast['0.5']
        upper = raw_forecast['0.9']
        lower = raw_forecast['0.1']
        point_forecast_df[model_name] = point_forecast
        upper_df[model_name] = upper
        lower_df[model_name] = lower

In [208]:
baseline_point = pd.read_csv('../baseline/dam_summary_pred.csv')#, parse_dates=['Date'])
baseline_point

,chronos2_8192_ARX_basic,chronosbolt_AR,chronos2_8192_AR,TimesFM_AR,price,price_decile,LEAR,DNN,ens_tsfm,ens_tsfm_ml,...,chronos2_2048_ARX_de_lu,chronos2_1024_AR,chronos2_1024_ARX_basic,chronos2_2048_ARX_basic,chronos2_2048_AR,chronos2small_lora_AR_ver3,chronos2_lora_8192_ARX,chronos2small_lora_AR_train_2048,chronos2small_lora_AR,chronos2_lora_AR
0,0.451530,1.285957,0.342308,-1.732063,0.10,1,-3.704949,-2.598481,0.086933,-0.992616,...,0.302101,-0.381310,1.130570,0.420456,0.932976,1.957451,0.475731,0.661636,0.661636,0.835510
1,-1.560555,0.009575,-0.666992,-3.823929,0.01,1,-11.535979,-6.427956,-1.510475,-4.000973,...,-0.573746,-1.236465,-0.576042,-0.694923,0.244629,-0.011925,-0.827698,-0.297066,-0.297066,-0.318733
2,-1.948982,-1.298004,-0.427330,-4.881111,0.00,1,-14.719947,-11.462571,-2.138857,-5.789658,...,-0.753159,-1.154526,-0.957375,-0.955452,0.337433,-1.907371,-1.147522,-2.246758,-2.246758,-0.636551
3,-3.609711,-2.085915,-1.630852,-8.487259,-0.01,1,-18.346627,-12.476321,-3.953434,-7.772781,...,-2.204292,-1.951279,-2.378334,-2.416046,-0.431137,-2.308960,-2.565857,-3.054520,-3.054520,-2.060936
4,-3.551567,-2.226128,-2.081856,-8.944252,-0.03,1,-16.963710,-11.174688,-4.200951,-7.490367,...,-2.449257,-2.440201,-2.585724,-2.642189,-1.013756,-2.140801,-2.394005,-2.312439,-2.312439,-2.193459
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8779,116.582140,124.514830,120.537186,126.298310,81.30,7,97.899953,121.473579,121.983116,117.884333,...,118.347190,120.695625,121.852660,120.438480,124.396400,123.278080,123.456160,124.318726,124.318726,121.424210
8780,108.555920,112.713600,111.896290,116.506850,45.60,3,83.666728,112.203552,112.418165,107.590490,...,107.937420,108.790215,111.522350,110.913890,112.487946,114.069534,114.076570,114.368810,114.368810,112.130160
8781,100.928820,104.126230,103.863815,105.073140,17.60,2,83.242242,107.962624,103.498001,100.866145,...,99.755520,100.658300,103.463326,102.870850,104.407080,106.498276,105.661000,106.003494,106.003494,104.098335
8782,96.347275,99.613014,99.173050,99.216490,4.04,1,83.416296,103.536430,98.587457,96.883759,...,96.375175,96.782234,100.560530,99.736480,100.833540,102.123990,102.000400,102.108740,102.108740,100.853740


In [209]:
correct_col = ['price', 'price_decile',
               'chronosbolt_AR',
               'TimesFM_AR', 
               'LEAR', 'DNN','ens_ml','persistent',
               'chronos2_1024_AR','chronos2_2048_AR','chronos2_8192_AR']

baseline_point = baseline_point[correct_col]


In [210]:
baseline_point.shape

(8784, 11)

In [211]:
point_forecast_df = pd.DataFrame(point_forecast_df)
point_forecast_df.index=raw_forecast[timestamp_col_parse]

In [212]:
point_forecast_df.reset_index(inplace=True)

In [213]:
point_forecast_df

,Date,timesfm25_1024_AR_neg,chronos2_8192_ARX_temporal,chronos2_2048_ARX_temporal,timesfm25_8192_AR_neg,chronos2_1024_ARX_temporal,timesfm25_2048_AR_neg
0,2023-01-01 00:00:00,-7.984528,2.128403,1.704681,-8.041138,4.075409,-5.420197
1,2023-01-01 01:00:00,-12.703583,0.423615,0.788330,-10.629303,3.331482,-8.998550
2,2023-01-01 02:00:00,-17.343414,-0.138901,-0.343445,-15.941437,2.273499,-13.923599
3,2023-01-01 03:00:00,-24.794678,-2.674332,-2.991730,-23.108398,0.941498,-19.640854
4,2023-01-01 04:00:00,-28.191680,-3.276321,-3.141632,-26.372192,0.962067,-22.672241
...,...,...,...,...,...,...,...
17539,2024-12-31 19:00:00,126.298280,99.956240,99.641620,127.122190,102.174500,125.531110
17540,2024-12-31 20:00:00,116.506820,87.685930,86.124245,117.867550,89.029140,116.505040
17541,2024-12-31 21:00:00,105.073105,75.887650,75.064380,106.783646,78.372475,106.104780
17542,2024-12-31 22:00:00,99.216480,72.960750,72.206000,102.145546,76.467350,101.872830


In [214]:
baseline_point = pd.concat([ point_forecast_df.iloc[-8784:,].reset_index(drop=True),baseline_point], axis=1)

In [215]:
baseline_point

,Date,timesfm25_1024_AR_neg,chronos2_8192_ARX_temporal,chronos2_2048_ARX_temporal,timesfm25_8192_AR_neg,chronos2_1024_ARX_temporal,timesfm25_2048_AR_neg,price,price_decile,chronosbolt_AR,TimesFM_AR,LEAR,DNN,ens_ml,persistent,chronos2_1024_AR,chronos2_2048_AR,chronos2_8192_AR
0,2024-01-01 00:00:00,-1.732063,-0.993057,0.158455,-1.811249,0.498779,-2.116943,0.10,1,1.285957,-1.732063,-3.704949,-2.598481,-3.151715,-3.91,-0.381310,0.932976,0.342308
1,2024-01-01 01:00:00,-3.823929,-3.589729,-1.903648,-3.909737,-1.571457,-3.696365,0.01,1,0.009575,-3.823929,-11.535979,-6.427956,-8.981967,-8.37,-1.236465,0.244629,-0.666992
2,2024-01-01 02:00:00,-4.881111,-5.421745,-3.554909,-3.852104,-2.527191,-4.213196,0.00,1,-1.298004,-4.881111,-14.719947,-11.462571,-13.091259,-10.00,-1.154526,0.337433,-0.427330
3,2024-01-01 03:00:00,-8.487267,-7.023384,-5.301277,-6.718201,-4.438095,-7.082481,-0.01,1,-2.085915,-8.487259,-18.346627,-12.476321,-15.411474,-11.80,-1.951279,-0.431137,-1.630852
4,2024-01-01 04:00:00,-8.944267,-6.586365,-5.622032,-6.904182,-4.524193,-7.531990,-0.03,1,-2.226128,-8.944252,-16.963710,-11.174688,-14.069199,-11.12,-2.440201,-1.013756,-2.081856
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8779,2024-12-31 19:00:00,126.298280,99.956240,99.641620,127.122190,102.174500,125.531110,81.30,7,124.514830,126.298310,97.899953,121.473579,109.686766,105.66,120.695625,124.396400,120.537186
8780,2024-12-31 20:00:00,116.506820,87.685930,86.124245,117.867550,89.029140,116.505040,45.60,3,112.713600,116.506850,83.666728,112.203552,97.935140,102.56,108.790215,112.487946,111.896290
8781,2024-12-31 21:00:00,105.073105,75.887650,75.064380,106.783646,78.372475,106.104780,17.60,2,104.126230,105.073140,83.242242,107.962624,95.602433,94.63,100.658300,104.407080,103.863815
8782,2024-12-31 22:00:00,99.216480,72.960750,72.206000,102.145546,76.467350,101.872830,4.04,1,99.613014,99.216490,83.416296,103.536430,93.476363,99.04,96.782234,100.833540,99.173050


In [216]:
baseline_point.rename(columns={'TimesFM_AR':'timesfm25_1024_AR_neg'},inplace=True)

In [217]:
baseline_point.to_csv('dam_summary_point.csv',index=False)

In [218]:
upper_df = pd.DataFrame(upper_df)
upper_df.index=raw_forecast[timestamp_col_parse]
lower_df = pd.DataFrame(lower_df)
lower_df.index=raw_forecast[timestamp_col_parse]

In [219]:
lear_interval = pd.read_csv('QR_forecasts_2024.csv',parse_dates=['DateTime'])
lear_interval.columns

Index(['DateTime', 'QR_730_q0.1', 'QR_730_q0.9'], dtype='str')

In [220]:
lower_lear = lear_interval[['DateTime', 'QR_730_q0.1']].rename(columns={'DateTime':'Date','QR_730_q0.1':'LEAR'})
upper_lear = lear_interval[['DateTime', 'QR_730_q0.9']].rename(columns={'DateTime':'Date','QR_730_q0.9':'LEAR'})

In [221]:
upper_df = upper_df.reset_index().merge(upper_lear, left_on='Date', right_on = 'Date', how='left')
lower_df = lower_df.reset_index().merge(lower_lear, left_on='Date', right_on = 'Date', how='left')

# lower_df.merge(lower_lear, left_index=True, right_on = 'DateTime',how='left')
# upper_df

In [222]:
upper_df

,Date,timesfm25_1024_AR_neg,chronos2_8192_ARX_temporal,chronos2_2048_ARX_temporal,timesfm25_8192_AR_neg,chronos2_1024_ARX_temporal,timesfm25_2048_AR_neg,LEAR
0,2023-01-01 00:00:00,1.424744,13.265472,12.093796,3.633087,15.789917,2.236175,NaN
1,2023-01-01 01:00:00,-4.197144,11.090302,10.591034,2.985382,14.075897,-1.092133,NaN
2,2023-01-01 02:00:00,-7.964935,9.672455,8.554199,0.105362,11.732422,-4.916351,NaN
3,2023-01-01 03:00:00,-12.242798,9.977035,8.710022,-6.642227,11.733383,-9.832413,NaN
4,2023-01-01 04:00:00,-12.884521,9.460358,8.540222,-5.656998,10.277817,-9.660599,NaN
...,...,...,...,...,...,...,...,...
17539,2024-12-31 19:00:00,164.922710,123.828320,125.921570,157.701020,121.867004,159.554930,186.035
17540,2024-12-31 20:00:00,154.108050,111.881670,108.683440,146.794950,106.746290,149.245790,153.700
17541,2024-12-31 21:00:00,141.651730,99.086290,94.041850,134.401370,94.905685,137.638700,129.570
17542,2024-12-31 22:00:00,134.351680,95.582016,90.949875,128.320790,93.248505,131.431150,127.220


In [223]:
upper_df.to_csv('dam_summary_upper.csv',index=False)
lower_df.to_csv('dam_summary_lower.csv',index=False)

## IMB raw

In [224]:
dataset = 'imb'

_forecast_folder = os.path.join(forecast_folder,dataset)
point_forecast_df = {}
upper_df = {}
lower_df = {}
for fname in os.listdir(_forecast_folder):
    print(fname)
    if 'csv' in fname and 'chronos' in fname:
        model_name = fname.replace('.csv','')
        timestamp_col_parse = 'cutoff_dates'
        raw = pd.read_csv(os.path.join(_forecast_folder, fname), parse_dates=[timestamp_col_parse])
        raw['QH'] = raw.index % 8 + 1
        raw = raw.sort_values(['cutoff_dates','QH']).reset_index(drop=True)
        raw['timestamp'] = raw['cutoff_dates'] + pd.to_timedelta((raw['QH'] - 1) * 15, unit='min')    
        raw.reset_index(drop=True)
        point_forecast_df[model_name] = raw['0.5'].iloc[:280256]
        
        upper_df[model_name] = raw['0.9'].iloc[:280256]
        lower_df[model_name] = raw['0.1'].iloc[:280256]

    if 'csv' in fname and 'timesfm' in fname:        
        
        model_name = fname.replace('.csv','')
        timestamp_col_parse = 'timestamp'
        # raw['QH'] = raw.index % 8 + 1
        point_forecast_df[model_name] = raw['0.5'].iloc[:280256]
        upper_df[model_name] = raw['0.9'].iloc[:280256]
        lower_df[model_name] = raw['0.1'].iloc[:280256]

chronos2_2048_ARX_temporal.csv
timesfm25_8192_AR_neg.csv
chronos2_1024_AR.csv
chronos2_1024_ARX_temporal.csv
timesfm25_2048_AR_neg.csv
chronos2_512_ARX_temporal.csv
chronos2_2048_AR.csv


In [225]:
upper_df = pd.DataFrame(upper_df)

upper_df['QH'] = upper_df.index % 8 + 1
upper_df['timestamp'] = raw['timestamp'].iloc[:280256]
upper_df.to_csv('imb_summary_upper.csv',index=False)

In [226]:
lower_df = pd.DataFrame(lower_df)

lower_df['QH'] = lower_df.index % 8 + 1
lower_df['timestamp'] = raw['timestamp'].iloc[:280256]
lower_df.to_csv('imb_summary_lower.csv',index=False)

In [227]:
baseline_df = pd.read_csv('../baseline/imb_pred.df.csv')
baseline_df['timestamp'] = raw['timestamp'].iloc[:280256]
correct_cols = ['timestamp','price','QH','chronosbolt_AR','chronos2_AR','TimesFM_AR',
                'DNN_2023_large', 'XGB_2023_medium', 'DNN_2023_small',
               'LEAR_2023_medium', 'DNN_2023_medium', 'LEAR_2023_large',
               'LEAR_2023_small', 'XGB_2023_large', 'XGB_2023_small', 'QH','ens_ml']
baseline_df = baseline_df[correct_cols]
baseline_df = pd.concat([baseline_df,pd.DataFrame(point_forecast_df)],axis=1)


In [228]:
baseline_df.rename(columns={'TimesFM_AR':'timesfm25_1024_AR_neg'},inplace=True)
baseline_df.to_csv('imb_summary_point.csv',index=False)
